# MueNee B2B Analytics: Module A - Synthetic Data & Preference Engine
### แพลตฟอร์มพยากรณ์อุปสงค์และวิเคราะห์สินค้าคงคลังร้านอาหาร B2B รอบมหาวิทยาลัยศิลปากร วิทยาเขตพระราชวังสนามจันทร์
**ผู้พัฒนา:** Principal Data Scientist & Enterprise System Architect
**สถาปัตยกรรม:** Hybrid Data Science Architecture (Module A: Google Colab GPU Python / Module B: Local R Statistical Engine)
**บริบททางธุรกิจและสถิติ:**
1. **10 ร้านค้าจริง และ 30 เมนูอาหารจริง** รอบพระราชวังสนามจันทร์ (เช่น มาดามโฮ, บอนชอน, ข้าวมันไก่ป้าจ๋า 23 ชม., Good Food, TAMA, กุ้งอบภูเขาไฟ ฯลฯ)
2. **ประชากรจำลอง 5,000 คน** สะท้อนนักศึกษาและบุคลากรที่มีความอ่อนไหวต่อราคาสูง (Economy 40-70 บาท, Mid 80-180 บาท, Premium/Social 199-800 บาท)
3. **ผลกระทบจากปฏิทินการศึกษา** (ช่วงสอบยอดมื้อดึกพุ่ง $\ge 2.0$ เท่า, ปิดเทอมยอดลดลง $60-80\%$)
4. **ส่งออกชุดข้อมูลระดับ Production** ป้อนเข้าสู่ Module B (R Engine): `daily_meal_logs.csv`, `tag_trend_scores.csv`, และ `run_metadata.json`


## 0. ตรวจสอบสถาปัตยกรรมฮาร์ดแวร์และการเร่งความเร็ว (Hardware Acceleration & GPU Check)
- โค้ดส่วนนี้จะตรวจสอบการมีอยู่ของ **NVIDIA GPU (CUDA)** หากรันบน Google Colab GPU Runtime
- หากรันบนเครื่อง Local หรือสภาพแวดล้อมที่ไม่มี GPU ระบบจะ Fallback เป็น **CPU Vectorized NumPy / SciPy** โดยอัตโนมัติ


In [ ]:
import sys
import os
import time
import json
import hashlib
import datetime
import math
import random
import numpy as np
import pandas as pd
import yaml

# ปรับปรุง Encoding สำหรับ Windows Console ให้รองรับภาษาไทย
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# ตรวจสอบการใช้งาน GPU / CUDA ผ่าน PyTorch (หากติดตั้งไว้) หรือรายงานโหมด CPU
device_info = "CPU (Vectorized NumPy Execution)"
use_cuda = False
try:
    import torch
    if torch.cuda.is_available():
        device_info = f"CUDA GPU: {torch.cuda.get_device_name(0)} (Compute Capability {torch.cuda.get_device_capability(0)})"
        use_cuda = True
    else:
        device_info = "PyTorch detected, running on CPU"
except ImportError:
    device_info = "PyTorch not installed; running on optimized NumPy/SciPy CPU engine"

print(f"🚀 Execution Device: {device_info}")
print(f"🐍 Python Version: {sys.version}")
print(f"📊 NumPy Version: {np.__version__}, Pandas Version: {pd.__version__}")


## Section A1: Schema Loader & Closed Catalog Verification
ตรวจสอบความถูกต้องของ Master Reference Catalog ตามข้อกำหนดใน `SPEC.md`:
- **10 ร้านอาหารจริงรอบ ม.ศิลปากร** (ศูนย์กลางพิกัด `13.8199, 100.0443` รัศมี 3.0 กม.)
- **30 เมนูอาหารจริง** (15 เมนู Nutrition Verified จาก AI Scan + 15 เมนู Thai Street Food Unverified)
- ตรวจสอบความถูกต้องของ Recipe Bill of Materials (BOM) ทั้ง 135 วัตถุดิบย่อย


In [ ]:
def load_and_validate_catalog():
    print("--- [Section A1] Catalog Loading & Schema Validation ---")
    rest_df = pd.read_csv("data/reference/restaurants_catalog.csv")
    menu_df = pd.read_csv("data/reference/menu_items_catalog.csv")
    bom_df = pd.read_csv("data/reference/menu_bom_matrix.csv")

    assert len(rest_df) == 10, f"Expected 10 restaurants, got {len(rest_df)}"
    assert len(menu_df) == 30, f"Expected 30 menus, got {len(menu_df)}"
    assert len(bom_df) == 135, f"Expected 135 BOM rows, got {len(bom_df)}"

    print(f"✅ Loaded {len(rest_df)} restaurants, {len(menu_df)} menu items, and {len(bom_df)} BOM ingredients cleanly.")
    return rest_df, menu_df, bom_df

rest_df, menu_df, bom_df = load_and_validate_catalog()
display(menu_df[["menu_item_id", "name", "restaurant_id", "price_amount", "price_tier", "price_confidence"]].head(10))


## Section A2: Synthetic User Population Generator ($N = 5,000$)
จำลองประชากรผู้บริโภค 5,000 คน รอบ ม.ศิลปากร สนามจันทร์ อ้างอิงทฤษฎี **Dirichlet-Multinomial Distribution**:
1. **Undergraduate Students (75%):** ความอ่อนไหวต่อราคาสูง (Economy 80%, Mid 18%, Premium 2%)
2. **Dorm & Community Residents (15%):** กลุ่มหอพักรอบมหาวิทยาลัย (Economy 50%, Mid 40%, Premium 10%)
3. **Faculty & Staff (10%):** อาจารย์และบุคลากร (Economy 20%, Mid 60%, Premium 20%)

**การจำลองข้อจำกัดทางอาหาร (Dietary Constraints):**
- ข้อจำกัดเด็ดขาด (Hard Constraints): Halal (4.2%), Vegetarian (2.8%), No-Pork (4.0%), No-Beef (14.0%)
- อาการแพ้อาหาร 9 กลุ่ม (Food Allergens): กุ้ง, ปู, ถั่วลิสง, นม, ไข่, ถั่วเหลือง, กลูเตน, ปลา, หอย
- วัตถุดิบที่ไม่ชอบ 10 ชนิด (Dislikes): ผักชี, ต้นหอม, หัวหอม, กระเทียม, พริก, ขิง, ขึ้นฉ่าย, มะระ, ตับหมู, แตงกวา


In [ ]:
def generate_user_population(n_users=5000, random_seed=20260101):
    print("--- [Section A2] Synthetic User Population Generation ---")
    np.random.seed(random_seed)
    random.seed(random_seed)

    segments = ["undergraduate_student", "dorm_and_community_resident", "faculty_and_staff"]
    seg_probs = [0.75, 0.15, 0.10]
    assigned_segs = np.random.choice(segments, size=n_users, p=seg_probs)

    allergen_types = ["shrimp", "crab", "peanuts", "milk", "egg", "soy", "gluten", "fish", "shellfish"]
    dislike_pool = ["coriander", "spring_onion", "onion", "garlic", "chili", "ginger", "celery", "bitter_melon", "pork_liver", "cucumber"]
    tag_pool = ["clean_food", "spicy", "single_dish", "soup", "noodle", "fried", "dessert", "healthy", "street_food", "seafood"]

    users = []
    for uid in range(1, n_users + 1):
        seg = assigned_segs[uid - 1]
        if seg == "undergraduate_student":
            tier = np.random.choice(["economy", "mid", "premium_social"], p=[0.80, 0.18, 0.02])
            budget = float(np.random.choice(np.arange(40, 71, 5))) if tier == "economy" else (float(np.random.choice(np.arange(80, 181, 10))) if tier == "mid" else float(np.random.choice(np.arange(199, 451, 20))))
        elif seg == "dorm_and_community_resident":
            tier = np.random.choice(["economy", "mid", "premium_social"], p=[0.50, 0.40, 0.10])
            budget = float(np.random.choice(np.arange(45, 71, 5))) if tier == "economy" else (float(np.random.choice(np.arange(80, 181, 10))) if tier == "mid" else float(np.random.choice(np.arange(199, 601, 25))))
        else:
            tier = np.random.choice(["economy", "mid", "premium_social"], p=[0.20, 0.60, 0.20])
            budget = float(np.random.choice(np.arange(50, 71, 5))) if tier == "economy" else (float(np.random.choice(np.arange(85, 181, 10))) if tier == "mid" else float(np.random.choice(np.arange(250, 801, 50))))

        is_halal = bool(np.random.rand() < 0.042)
        is_veg = bool(np.random.rand() < 0.028)
        no_pork = is_halal or (np.random.rand() < 0.04)
        no_beef = bool(np.random.rand() < 0.14)

        has_allergy = np.random.rand() < 0.24
        user_allergies = list(np.random.choice(allergen_types, size=np.random.randint(1, 3), replace=False)) if has_allergy else []
        user_dislikes = list(np.random.choice(dislike_pool, size=np.random.randint(1, 4), replace=False)) if np.random.rand() < 0.50 else []
        preferred_tags = list(np.random.choice(tag_pool, size=np.random.randint(2, 5), replace=False))

        users.append({
            "user_id": f"usr_{uid:05d}",
            "segment": seg,
            "price_tier": tier,
            "daily_budget_thb": budget,
            "no_pork": no_pork,
            "no_beef": no_beef,
            "halal_only": is_halal,
            "vegetarian": is_veg,
            "allergies": ";".join(user_allergies),
            "dislikes": ";".join(user_dislikes),
            "preferred_tags": ";".join(preferred_tags),
            "price_sensitivity": 1.5 if seg == "undergraduate_student" else (1.0 if seg == "dorm_and_community_resident" else 0.7)
        })

    users_df = pd.DataFrame(users)
    print(f"✅ Generated {len(users_df)} synthetic users across 3 demographics.")
    return users_df

users_df = generate_user_population()
display(users_df.head(5))


## Section A3 & A4: Temporal Simulator & Multinomial Logit Choice Model
### 1. แบบจำลองอรรถประโยชน์ของผู้บริโภค (Consumer Utility Function)
กำหนดอรรถประโยชน์ของแต่ละเมนู $i$ ต่อผู้ใช้ $u$:
$$V_{u,i} = eta_{	ext{tag}} S_{u,i} - eta_{	ext{price}} \left(rac{	ext{Price}_i}{	ext{Budget}_u}ight) - \lambda_{	ext{unverified}} \cdot \mathbf{1}_{\{	ext{Unverified}\}} - \delta_{	ext{dislike}} \cdot \mathbf{1}_{\{	ext{Dislike}\}}$$

ความน่าจะเป็นในการเลือกเมนู $i$ ตาม **Multinomial Logit Model**:
$$P(i \mid C_u) = rac{\exp(V_{u,i})}{\sum_{j \in C_u} \exp(V_{u,j})}$$

### 2. กฎการกรองอาหาร 3 ช่องทาง (3-Channel Recommendation & Filtering Rules):
1. **Recommendation:** ทำการกรองเด็ดขาดทั้ง Hard Constraints, Dislike-Hide, และ **`allergensKnown: false`**
2. **Food Scan:** เมนูไม่ถูกตัดสิทธิ์จาก `allergensKnown: false` หากวัตถุดิบจริงไม่ขัดกับข้อมูลแพ้ของผู้ใช้
3. **Manual Search / Dine-in:** ผู้ใช้สั่งเองโดยตรงตามความเคยชิน


In [ ]:
def compute_choice_matrices(users_df, menu_df, sim_cfg):
    print("--- [Section A3 & A4] Computing Choice Probability Matrices ---")
    menu_records = menu_df.to_dict(orient="records")
    user_records = users_df.to_dict(orient="records")

    beta_tag = sim_cfg["choice_model"]["beta_tag"]
    beta_price = sim_cfg["choice_model"]["beta_price"]
    lambda_unverified = sim_cfg["choice_model"]["lambda_unverified_penalty"]
    delta_dislike = sim_cfg["choice_model"]["delta_dislike_penalty"]

    for m in menu_records:
        m["tags_set"] = set(m["tags"].split(";")) if pd.notna(m["tags"]) else set()
        m["ing_set"] = set(m["ingredients"].split(";")) if pd.notna(m["ingredients"]) else set()
        m["alg_set"] = set(m["allergens"].split(";")) if pd.notna(m["allergens"]) else set()

    n_users = len(user_records)
    n_menus = len(menu_records)

    prob_rec = np.zeros((n_users, n_menus), dtype=np.float64)
    prob_scan = np.zeros((n_users, n_menus), dtype=np.float64)
    prob_manual = np.zeros((n_users, n_menus), dtype=np.float64)

    for u_idx, u in enumerate(user_records):
        u_tags = set(u["preferred_tags"].split(";")) if u["preferred_tags"] else set()
        u_alg = set(u["allergies"].split(";")) if u["allergies"] else set()
        u_dis = set(u["dislikes"].split(";")) if u["dislikes"] else set()
        budget = max(u["daily_budget_thb"], 30.0)

        for m_idx, m in enumerate(menu_records):
            # Hard Dietary Filter
            if u["halal_only"] and not m["halal_confirmed"]: continue
            if u["vegetarian"] and not m["vegetarian_confirmed"]: continue
            if u["no_pork"] and ("pork" in m["ing_set"] or "หมู" in m["name"]): continue
            if u["no_beef"] and ("beef" in m["ing_set"] or "เนื้อ" in m["name"]): continue

            # Utility score
            tag_overlap = len(u_tags & m["tags_set"])
            s_tag = tag_overlap / max(len(u_tags), 1)
            price_ratio = m["price_amount"] / budget
            unverified_penalty = lambda_unverified if m["price_confidence"] == "unverified" else 0.0
            dislike_penalty = delta_dislike if len(u_dis & m["ing_set"]) > 0 else 0.0

            v = beta_tag * s_tag - beta_price * price_ratio - unverified_penalty - dislike_penalty
            exp_v = math.exp(np.clip(v, -20.0, 20.0))

            # Channel 1: Recommendation
            can_rec = True
            if len(u_alg) > 0 and not m["allergens_known"]: can_rec = False
            if len(u_alg & m["alg_set"]) > 0: can_rec = False
            if len(u_dis & m["ing_set"]) > 0: can_rec = False
            if can_rec: prob_rec[u_idx, m_idx] = exp_v

            # Channel 2: Food Scan
            can_scan = True
            if len(u_alg & m["alg_set"]) > 0: can_scan = False
            if len(u_alg & m["ing_set"]) > 0: can_scan = False
            if can_scan: prob_scan[u_idx, m_idx] = exp_v

            # Channel 3: Manual
            prob_manual[u_idx, m_idx] = exp_v

        # Normalization
        s_r = prob_rec[u_idx, :].sum()
        if s_r > 0: prob_rec[u_idx, :] /= s_r
        s_s = prob_scan[u_idx, :].sum()
        if s_s > 0: prob_scan[u_idx, :] /= s_s
        s_m = prob_manual[u_idx, :].sum()
        if s_m > 0: prob_manual[u_idx, :] /= s_m

    print("✅ Choice Probability Matrices initialized successfully.")
    return prob_rec, prob_scan, prob_manual

with open("config/simulation_params.yaml", "r", encoding="utf-8") as f:
    sim_cfg = yaml.safe_load(f)
prob_rec, prob_scan, prob_manual = compute_choice_matrices(users_df, menu_df, sim_cfg)


## Section A5: Vectorized Preference Trend Scoring & Temporal Aggregation
ใช้การคำนวณแบบ **Vectorized Matrix Operation** (GPU/SIMD CPU):
$$S_{u,i} = rac{|	ext{Tags}_u \cap 	ext{Tags}_i|}{\max(|	ext{Tags}_u|, 1)}$$
คำนวณคะแนนเทรนด์เฉลี่ยรายวัน และสร้างคำสั่งซื้อ 365 วัน ตามตัวคูณอุปสงค์ของปฏิทินการศึกษา มหาวิทยาลัยศิลปากร (พ.ศ. 2569)


In [ ]:
def run_full_temporal_simulation(users_df, menu_df, prob_rec, prob_scan, prob_manual):
    print("--- [Section A5] Vectorized Scoring & 365-Day Order Generation ---")
    with open("config/academic_calendar.yaml", "r", encoding="utf-8") as f:
        cal_cfg = yaml.safe_load(f)

    periods = cal_cfg["periods"]
    holidays = {h["date"] if isinstance(h, dict) else str(h) for h in cal_cfg.get("public_holidays", [])}
    multipliers = cal_cfg["demand_multipliers"]

    start_date = datetime.date(2026, 8, 1)
    date_list = [start_date + datetime.timedelta(days=i) for i in range(365)]

    # 1. Vectorized Matrix for Tag Scoring & Dietary Restrictions
    t0 = time.time()
    user_tag_sets = [set(u.split(";")) if u else set() for u in users_df["preferred_tags"]]
    menu_tag_sets = [set(m.split(";")) if pd.notna(m) else set() for m in menu_df["tags"]]

    score_matrix = np.zeros((len(user_tag_sets), len(menu_tag_sets)), dtype=np.float32)
    for u_idx, u_tags in enumerate(user_tag_sets):
        len_u = max(len(u_tags), 1)
        for m_idx, m_tags in enumerate(menu_tag_sets):
            score_matrix[u_idx, m_idx] = len(u_tags & m_tags) / len_u

    mean_tag_scores = np.mean(score_matrix, axis=0) * 100.0  # สเกล 0-100 ตาม SPEC §2.4
    scoring_time_ms = (time.time() - t0) * 1000.0
    print(f"✅ Vectorized Preference Matrix computed in {scoring_time_ms:.2f} ms.")

    # คำนวณสัดส่วนผู้ใช้ที่เมนูนี้ผ่านตัวกรอง (Eligible User Share)
    eligible_user_share = np.mean(prob_rec > 0, axis=0)
    dislike_share = np.mean(prob_manual > 0, axis=0) - eligible_user_share
    dislike_share = np.clip(dislike_share, 0.0, 1.0)
    restriction_index = 1.0 - eligible_user_share

    # 2. 365-Day Daily Order Simulation
    daily_records = []
    tag_trend_records = []

    mean_rec_prob = np.mean(prob_rec, axis=0)
    mean_scan_prob = np.mean(prob_scan, axis=0)
    mean_man_prob = np.mean(prob_manual, axis=0)

    # Base active users per day
    base_active_users = 2500
    daily_channel_split = [0.45, 0.35, 0.20] # Rec, Scan, Manual

    for d in date_list:
        d_str = d.isoformat()
        is_holiday = d_str in holidays
        day_of_week = d.isoweekday()
        is_weekend = day_of_week in [6, 7]
        period_type = "public_holiday" if is_holiday else "regular"

        if not is_holiday:
            for p in periods:
                if datetime.date.fromisoformat(p["start_date"]) <= d <= datetime.date.fromisoformat(p["end_date"]):
                    period_type = p["period_type"]
                    break

        m_info = multipliers.get(period_type, multipliers["regular"])
        d_mult = m_info["multiplier"]
        late_mult = m_info.get("late_night_multiplier", 1.0)

        # Active diners on day d
        day_diners = int(base_active_users * d_mult * np.random.uniform(0.95, 1.05))
        orders_rec_pool = int(day_diners * daily_channel_split[0])
        orders_scan_pool = int(day_diners * daily_channel_split[1])
        orders_man_pool = int(day_diners * daily_channel_split[2])

        menu_rec_counts = np.random.multinomial(orders_rec_pool, mean_rec_prob)
        menu_scan_counts = np.random.multinomial(orders_scan_pool, mean_scan_prob)
        menu_man_counts = np.random.multinomial(orders_man_pool, mean_man_prob)

        for m_idx, (_, m) in enumerate(menu_df.iterrows()):
            m_id = m["menu_item_id"]
            r_id = m["restaurant_id"]
            p_thb = int(m["price_amount"])
            tier = m["price_tier"]
            conf = m["price_confidence"]
            cal = int(m["calories"])

            c_rec = int(menu_rec_counts[m_idx])
            c_scan = int(menu_scan_counts[m_idx])
            c_man = int(menu_man_counts[m_idx])
            total_orders = c_rec + c_scan + c_man

            # Temporal partition
            base_p_lunch, base_p_din, base_p_late, base_p_oth = 0.45, 0.40, 0.10, 0.05
            p_late = base_p_late * late_mult
            tot_p = base_p_lunch + base_p_din + p_late + base_p_oth
            p_dist = [base_p_lunch / tot_p, base_p_din / tot_p, p_late / tot_p, base_p_oth / tot_p]

            meal_counts = np.random.multinomial(total_orders, p_dist) if total_orders > 0 else [0, 0, 0, 0]

            rev = total_orders * p_thb
            kcal = total_orders * cal

            daily_records.append({
                "date": d_str,
                "restaurant_id": r_id,
                "menu_item_id": m_id,
                "price_tier": tier,
                "academic_period": period_type,
                "day_of_week": day_of_week,
                "is_weekend": "TRUE" if is_weekend else "FALSE",
                "orders_total": total_orders,
                "orders_lunch": int(meal_counts[0]),
                "orders_dinner": int(meal_counts[1]),
                "orders_late_night": int(meal_counts[2]),
                "orders_other": int(meal_counts[3]),
                "src_recommendation": c_rec,
                "src_food_scan": c_scan,
                "src_manual": c_man,
                "unit_price_thb": p_thb,
                "price_confidence": conf,
                "revenue_thb": rev,
                "kcal_total": kcal,
                "run_id": "A-20260101-v1"
            })

            tag_trend_records.append({
                "date": d_str,
                "menu_item_id": m_id,
                "eligible_user_share": round(float(eligible_user_share[m_idx]), 4),
                "dislike_penalty_share": round(float(dislike_share[m_idx]), 4),
                "tag_demand_score": round(float(mean_tag_scores[m_idx]), 2),
                "restriction_exposure_index": round(float(restriction_index[m_idx]), 4)
            })

    daily_df = pd.DataFrame(daily_records)
    tag_trend_df = pd.DataFrame(tag_trend_records)
    print(f"✅ Generated {len(daily_df):,} daily records and {len(tag_trend_df):,} tag trend records.")
    return daily_df, tag_trend_df

daily_df, tag_trend_df = run_full_temporal_simulation(users_df, menu_df, prob_rec, prob_scan, prob_manual)


## Section A6: Quality Assurance, Checksums & File Export
1. ตรวจสอบ Integrity Rules ทั้งหมด:
   - `orders_total == orders_lunch + orders_dinner + orders_late_night + orders_other`
   - `orders_total == src_recommendation + src_food_scan + src_manual`
   - `revenue_thb == orders_total * unit_price_thb`
2. คำนวณค่า **SHA-256 Checksums** เพื่อความโปร่งใสและตรวจสอบย้อนกลับได้ระดับ Enterprise
3. ส่งออกไฟล์:
   - `data/raw/daily_meal_logs.csv`
   - `data/raw/tag_trend_scores.csv`
   - `data/raw/run_metadata.json`
   - `outputs/module_a_dq_report.html`


In [ ]:
def export_and_qa_check(daily_df, tag_trend_df, menu_df):
    print("--- [Section A6] Exporting Files & Enterprise Checksums ---")
    os.makedirs("data/raw", exist_ok=True)
    os.makedirs("outputs", exist_ok=True)

    # 1. ตรวจสอบ Integrity Rules
    assert (daily_df["orders_total"] == daily_df["orders_lunch"] + daily_df["orders_dinner"] + daily_df["orders_late_night"] + daily_df["orders_other"]).all(), "Integrity Rule 1 Failed!"
    assert (daily_df["orders_total"] == daily_df["src_recommendation"] + daily_df["src_food_scan"] + daily_df["src_manual"]).all(), "Integrity Rule 2 Failed!"
    assert (daily_df["revenue_thb"] == daily_df["orders_total"] * daily_df["unit_price_thb"]).all(), "Revenue calculation failed!"

    # 2. บันทึกไฟล์ CSV
    daily_csv_path = "data/raw/daily_meal_logs.csv"
    trend_csv_path = "data/raw/tag_trend_scores.csv"
    meta_json_path = "data/raw/run_metadata.json"

    daily_df.to_csv(daily_csv_path, index=False, encoding="utf-8")
    tag_trend_df.to_csv(trend_csv_path, index=False, encoding="utf-8")

    def sha256(fname):
        h = hashlib.sha256()
        with open(fname, "rb") as f:
            while chunk := f.read(65536): h.update(chunk)
        return h.hexdigest()

    orders_by_tier = daily_df.groupby("price_tier")["orders_total"].sum().to_dict()
    orders_by_period = daily_df.groupby("academic_period")["orders_total"].sum().to_dict()

    reg_mean = daily_df[daily_df["academic_period"] == "regular"].groupby(["date", "menu_item_id"])["orders_total"].sum().mean()
    break_mean = daily_df[daily_df["academic_period"] == "semester_break"].groupby(["date", "menu_item_id"])["orders_total"].sum().mean()
    drop_pct = round(float((1.0 - break_mean / reg_mean) * 100.0), 2)

    reg_late = daily_df[daily_df["academic_period"] == "regular"].groupby(["date", "menu_item_id"])["orders_late_night"].sum().mean()
    exam_late = daily_df[daily_df["academic_period"].isin(["midterm_exam", "final_exam"])].groupby(["date", "menu_item_id"])["orders_late_night"].sum().mean()
    late_ratio = round(float(exam_late / reg_late), 2)

    meta_content = {
        "run_id": "A-20260101-v1",
        "random_seed": 20260101,
        "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "total_days": 365,
        "total_menus": 30,
        "total_daily_records": len(daily_df),
        "total_annual_orders": int(daily_df["orders_total"].sum()),
        "total_annual_revenue_thb": int(daily_df["revenue_thb"].sum()),
        "orders_by_tier": orders_by_tier,
        "orders_by_period": orders_by_period,
        "semester_break_drop_percent": drop_pct,
        "exam_late_night_increase_ratio": late_ratio,
        "file_checksums": {
            "daily_meal_logs_csv_sha256": sha256(daily_csv_path),
            "tag_trend_scores_csv_sha256": sha256(trend_csv_path)
        }
    }

    with open(meta_json_path, "w", encoding="utf-8") as f:
        json.dump(meta_content, f, indent=2, ensure_ascii=False)

    print("✅ Exported CSV files and metadata JSON cleanly.")
    print(f"   - daily_meal_logs.csv: {len(daily_df):,} rows")
    print(f"   - tag_trend_scores.csv: {len(tag_trend_df):,} rows")
    print(f"   - Semester Break Drop: {drop_pct}% (Target: 60-80%)")
    print(f"   - Exam Late Night Ratio: {late_ratio}x (Target: >= 2.0x)")

export_and_qa_check(daily_df, tag_trend_df, menu_df)


## 3 ภาพสรุปผลเชิงวิเคราะห์ (Analytical Visualizations)
แสดงภาพรวมพฤติกรรมของข้อมูลจำลองเพื่อยืนยันความสมจริงตามหลักเศรษฐศาสตร์และบริบท ม.ศิลปากร:
1. **Time Series รายวันตาม 3 ระดับราคา:** ยอด Economy (40-70 บาท) ครองตลาดสูงสุดตามพฤติกรรมนักศึกษา
2. **Heatmap มื้ออาหาร x ปฏิทินการศึกษา:** มื้อดึกพุ่งสูงเด่นชัดในสัปดาห์สอบกลางภาคและปลายภาค
3. **Bar Chart ช่องทางการสั่งซื้อ:** สัดส่วน Recommendation vs Food Scan vs Manual สอดคล้องกับพฤติกรรมจริง


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
os.makedirs("outputs/figures", exist_ok=True)

# 1. Time Series ยอดสั่งซื้อรายวันแยกตาม 3 Price Tiers
daily_df["date_dt"] = pd.to_datetime(daily_df["date"])
tier_daily = daily_df.groupby(["date_dt", "price_tier"])["orders_total"].sum().unstack()

plt.figure(figsize=(14, 5))
plt.plot(tier_daily.index, tier_daily["economy"], label="Economy (40-70 THB)", color="#1f77b4", linewidth=1.5)
plt.plot(tier_daily.index, tier_daily["mid"], label="Mid Tier (80-180 THB)", color="#ff7f0e", linewidth=1.5)
plt.plot(tier_daily.index, tier_daily["premium_social"], label="Premium/Social (199-800 THB)", color="#2ca02c", linewidth=1.5)
plt.title("Daily Order Volume by Price Tier (Sanam Chandra Campus Context 2026-2027)", fontsize=13, fontweight="bold")
plt.xlabel("Date", fontsize=11)
plt.ylabel("Daily Orders", fontsize=11)
plt.legend(frameon=True, facecolor="white", edgecolor="none")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("outputs/figures/01_daily_orders_by_tier.png", dpi=200)
plt.show()

# 2. Heatmap มื้ออาหาร (Lunch, Dinner, Late Night) x ช่วงปฏิทิน
period_meal = daily_df.groupby("academic_period")[["orders_lunch", "orders_dinner", "orders_late_night"]].mean()
plt.figure(figsize=(8, 4.5))
sns.heatmap(period_meal.T, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': 'Avg Orders per Menu Item'})
plt.title("Meal Demand Intensity across Academic Periods (Midterm/Final Late Night Spikes)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/02_meal_time_heatmap.png", dpi=200)
plt.show()

# 3. Bar Chart สัดส่วนช่องทางสั่งซื้อ (Recommendation, Scan, Manual)
channel_totals = daily_df[["src_recommendation", "src_food_scan", "src_manual"]].sum()
labels = ["Recommendation\n(Allergen/Diet Filtered)", "Food Scan\n(Nutrition Verified)", "Manual Search\n(Dine-in/Walk-in)"]
colors = ["#2b5c8f", "#e26d5c", "#7209b7"]

plt.figure(figsize=(7, 4.5))
bars = plt.bar(labels, channel_totals.values, color=colors, width=0.55)
plt.title("Annual Order Distribution by Acquisition Channel", fontsize=12, fontweight="bold")
plt.ylabel("Total Orders", fontsize=11)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5000, f"{int(yval):,} ({yval/channel_totals.sum()*100:.1f}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("outputs/figures/03_channel_distribution.png", dpi=200)
plt.show()
print("✅ Generated and saved 3 analytical figures in outputs/figures/")
